# Реальные банковские источники и экспертная разметка

Корпус содержит снимки англоязычных FAQ Consumer Financial Protection Bureau, юрисдикция США. Это исследовательские материалы, не одобренная база конкретного банка. Одна статья помещена в карантин из-за противоречия с отдельным официальным обновлением; остальные требуют предметной проверки. Автоматически предложенные вопросы не являются экспертной разметкой.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/public_banking'
report = json.loads((ROOT / 'reports/public_banking_corpus.json').read_text(encoding='utf-8'))
print('Кандидаты:', report['candidates'], '| research:', report['research_eligible'], '| требуют пересмотра:', report['review_due'])
display(pd.DataFrame(report['articles']))
display(report['quarantined'])


## Просмотр исходного материала

Дата updated_at взята из last reviewed источника. retrieved_at — дата сбора, а не обновления смысла. Снимок относится к текстовому reader-представлению, не к оригинальному HTML. Сбор из сохранённых снимков воспроизводится командой `python scripts/build_public_banking.py` и проверяет SHA-256.


In [ ]:
articles = [json.loads(line) for line in (DATA / 'candidates.jsonl').read_text(encoding='utf-8').splitlines()]
SELECTED_ARTICLE_ID = 'cfpb_1017'
article = next(row for row in articles if row['id'] == SELECTED_ARTICLE_ID)
print(article['title'])
print(article['source'], '|', article['status'], '|', article['jurisdiction'])
print(article['content'])


## Проверка режима использования

По умолчанию загрузчик и API принимают только serving-материалы. Исследовательский доступ требует явного purpose=research. Статья из карантина не входит в research_articles.jsonl. Ни индексация, ни лицензионная доступность не заменяют экспертную проверку содержания.


In [ ]:
from src.rag.retriever import load_documents
try:
    load_documents(DATA / 'research_articles.jsonl')
except ValueError as error:
    print('Ожидаемая блокировка serving:', error)
research_articles = load_documents(DATA / 'research_articles.jsonl', purpose='research')
print('Исследовательский корпус:', len(research_articles))


## Проверка восстановления источника по его вопросу

Это sanity check на FAQ-заголовках издателя: нужно найти исходную статью по её опубликованному вопросу. Метрики относятся к известному первоисточнику, а не к полной экспертной релевантности. Они не измеряют качество клиентских ответов, не выбирают production-метод и не используют holdout.


In [ ]:
probe = json.loads((ROOT / 'reports/public_banking_probe.json').read_text(encoding='utf-8'))
display(pd.DataFrame(probe['metrics']).T)
print(probe['limitations'])


## Пакет для экспертной проверки

Заполните окончательные relevant_document_ids, query_group_id, split, annotation_status, reviewer_id, reviewer_role и reviewed_at в локальной копии шаблона. Reviewer ID используйте как псевдоним. Для holdout нужны новые human_authored вопросы и другой reviewer, не автор. Данные должны быть реально проверены: простая смена флагов не создаёт независимую оценку. По умолчанию предложения AI скрыты, чтобы снизить якорное смещение. Заполненные файлы храните в work, не в публичном Git.


In [ ]:
drafts = [json.loads(line) for line in (DATA / 'annotation_drafts.jsonl').read_text(encoding='utf-8').splitlines()]
SHOW_PROPOSED_LABELS = False
columns = ['query_id', 'query', 'language', 'annotation_status', 'split']
if SHOW_PROPOSED_LABELS:
    columns += ['proposed_relevant_document_ids', 'proposed_group_id']
display(pd.DataFrame(drafts)[columns])
from src.data.public_banking import validate_reviewed_queries
try:
    validate_reviewed_queries(drafts, {row['id'] for row in research_articles})
except ValueError as error:
    print('Экспертный gate ожидаемо закрыт:', error)


После реального review экспорт выполняется через scripts/export_banking_review.py с --input и новым --output-dir внутри work. Он проверяет fingerprint корпуса, статусы review, группы, дубли, источники и наличие answerable/unanswerable в обеих частях. Это формальная проверка аттестаций reviewer, а не проверка его личности или квалификации. Содержательную независимость необходимо обеспечить самим протоколом разметки.
